# Python Function operator in rocAL
This example demonstrates how to use `fn.python_function` to inject custom Python processing into a rocAL pipeline. We build a simple image pipeline and apply Python-based brightness, crop, and horizontal flip steps.

### Common Code

In [ ]:
import numpy as np
from functools import partial
import random
import os
import matplotlib.pyplot as plt
%matplotlib inline

import amd.rocal.fn as fn
import amd.rocal.types as types
from amd.rocal.pipeline import Pipeline
from amd.rocal.plugin.generic import ROCALClassificationIterator


### Configuring rocAL pipeline

In [ ]:
# Data directory
image_dir = "/opt/rocm/share/rocal/test/data/images/AMD-tinyDataSet/"

# Pipeline configuration
rocal_cpu = True  # python_function is only supported on Host backend
batch_size = 2
num_threads = 4
device_id = 0
seed = 2


### Defining Python functions

In [ ]:
def random_augmentation(probability, augmented, original):
    condition = random.random() < probability
    neg_condition = not condition
    return condition * augmented + neg_condition * original

def brightness_fn(img):
    # Apply random brightness scaling in [0.7, 1.3]
    brightness_scale = random_augmentation(0.5, random.uniform(0.7, 1.3), 1.0)
    return (img * brightness_scale).astype(np.uint8)  # Ensure uint8 output

def crop_fn(img, crop_size):
    # Crop along height and width (NHWC layout)
    return img[:, :crop_size[0], :crop_size[1], :]

def flip_fn(img):
    # Horizontal flip with 50% probability
    if random.random() < 0.5:
        return img[:, :, ::-1, :]
    else:
        return img

class NormalizeWithStats:
    def __init__(self, mean, std):
        self.mean = np.array(mean).reshape(1, 1, 1, -1)
        self.std = np.array(std).reshape(1, 1, 1, -1)
    def __call__(self, batch):
        # Normalize using user-provided mean and std
        return ((batch - self.mean) / self.std).astype(np.float32)

# Bind crop size via partial for convenience
crop_image_fn = partial(crop_fn, crop_size=(224, 224))
normalizer = NormalizeWithStats(mean=[0.485 * 255, 0.456 * 255, 0.406 * 255],
                                std=[0.229 * 255, 0.224 * 255, 0.225 * 255])


### Python function pipeline

We read images using the file reader and decode them. Then we chain `fn.python_function` calls to apply random brightness, crop to 224×224, and an optional horizontal flip. Finally, we demonstrate passing a callable class to normalize the batch.

In [ ]:
pipe = Pipeline(batch_size=batch_size, num_threads=num_threads, device_id=device_id, seed=seed, rocal_cpu=rocal_cpu, tensor_layout=types.NHWC)
with pipe:
    jpegs, _ = fn.readers.file(file_root=image_dir)
    images = fn.decoders.image(jpegs, file_root=image_dir, output_type=types.RGB, shard_id=0, num_shards=1, random_shuffle=False)
    rand_brightness = fn.python_function(images, function=brightness_fn, dtype=types.UINT8, layout=types.NHWC)
    cropped = fn.python_function(rand_brightness, function=crop_image_fn, output_dims=(224, 224, 3), dtype=types.UINT8, layout=types.NHWC)
    flipped = fn.python_function(cropped, function=flip_fn, dtype=types.UINT8, layout=types.NHWC)
    normalized = fn.python_function(flipped, function=normalizer, dtype=types.FLOAT, layout=types.NHWC)
    pipe.set_outputs(flipped, normalized)  # Return both for visualization

pipe.build()
data_loader = ROCALClassificationIterator(pipe)

### Visualizing outputs

We display both the flipped (uint8) output and the normalized (float) output for the same sample. For normalized data, we clip to [0, 255] for visualization.

In [ ]:
# Fetch a batch and visualize every sample from both outputs
batch = next(iter(data_loader))
flipped_batch = batch[0][0]
normalized_batch = batch[0][1]

batch_size = flipped_batch.shape[0]
fig, axes = plt.subplots(nrows=batch_size, ncols=2, figsize=(10, 5 * batch_size))

if batch_size == 1:
    axes = np.expand_dims(axes, axis=0)

for sample_idx in range(batch_size):
    flipped_img = flipped_batch[sample_idx].astype(np.uint8)
    axes[sample_idx, 0].imshow(flipped_img)
    axes[sample_idx, 0].set_title(f'flipped sample {sample_idx}')
    axes[sample_idx, 0].axis('off')

    normalized_img = normalized_batch[sample_idx]
    normalized_img_vis = np.clip(normalized_img * 255, 0, 255).astype(np.uint8)
    axes[sample_idx, 1].imshow(normalized_img_vis)
    axes[sample_idx, 1].set_title(f'normalized (clipped) sample {sample_idx}')
    axes[sample_idx, 1].axis('off')

data_loader.reset()
plt.tight_layout()
plt.show()